<a href="https://colab.research.google.com/github/HectorNieto00/Machine-Learning-Models/blob/main/CreditCard_Frauds.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Task 2 – Exploratory Data Analysis (EDA)**  
In this section, an exploratory analysis of the credit card fraud dataset is performed.  
The objective is to understand the structure of the data, detect patterns, identify anomalies, and investigate correlations, following research-based best practices for fraud detection analysis.


**Loading the Dataset**

The dataset is imported from an Excel file and loaded into a Pandas DataFrame called `df`. This dataset contains anonymised transaction features (V1–V28), transaction time, amount, and a target variable `Class` indicating whether a transaction is fraudulent (1) or legitimate (0).


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_excel('creditcard.xlsx')
df.head()

In [ ]:
df.tail()

**Dataset Shape**

The dataset contains **284,807 rows** and **31 columns**.  
Each row represents a single credit card transaction.  
The 31 columns include:
- 28 PCA-transformed anonymised features (V1–V28)
- Time
- Amount
- Class (target variable)

A large sample size helps ensure reliable model training and reduces variance.


In [ ]:
df.shape

**Dataset Information**

This output shows:
- All **31 columns** contain **no missing values**
- `Time` and `Class` are integers
- All other variables are floats

This is ideal for ML because it requires no missing-value imputation and the numerical format is compatible with most algorithms. The PCA-transformed variables have no semantic meaning but preserve variance from the original features.


In [ ]:
df.info()

**Descriptive Statistics**

The descriptive summary reveals:
- PCA components (V1–V28) have a mean close to 0 due to PCA standardisation.
- `Amount` shows a large variance, with a maximum of 25,691, indicating the presence of high-value transactions.
- `Class` has a mean of 0.0017, highlighting a strong class imbalance  
  (fraud cases ≈ 0.17% of the dataset).

Understanding these distributions is essential before modelling because  
imbalanced data requires special handling (SMOTE, stratification).


In [ ]:
df.describe().T

**Class Distribution**

Fraud detection datasets are known for extreme class imbalance. Visualising the distribution of the target variable helps quantify the imbalance and highlights the need for resampling techniques during modelling.

0 (Legitimate) has a proportion of 99.827251% and 1 (Fraud) only 0.172749%. This means that only 0.17% of all transactions are fraudulent, confirming an extreme imbalance.
Such skewed distribution indicates that standard machine learning models may become biased toward predicting the majority class. Therefore, resampling strategies such as SMOTE, undersampling, or cost-sensitive learning will be essential to improve model performance.


In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(data=df, x='Class')
plt.title("Class Distribution (0 = Non-Fraud, 1 = Fraud)")
plt.show()

df['Class'].value_counts(normalize=True)*100


**Histograms of Numerical Features**

Histograms allow inspection of variable distributions. Since V1–V28 are PCA-generated, their distributions are centred near zero. The 'Amount' variable shows right-skewed behaviour, typical of real-world transactions.


In [ ]:
df.hist(figsize=(20,20), bins=30)
plt.suptitle("Distribution of All Numerical Variables", fontsize=20)
plt.show()


**Boxplot for Transaction Amount**

A boxplot helps visualise extreme values. High-value transactions may indicate anomalies or fraudulent behaviour, but they may also be legitimate. A more detailed analysis is needed later to understand their impact on the model.


In [ ]:
plt.figure(figsize=(8,4))
sns.boxplot(x=df['Amount'])
plt.title("Boxplot of Transaction Amount")
plt.show()


In [ ]:
data = df['Amount']

minimum = np.min(data)
q1 = np.percentile(data, 25)
median = np.median(data)
q3 = np.percentile(data, 75)
maximum = np.max(data)
iqr = q3 - q1

# Limits to detect outliers
lower_bound = q1 - 1.5*iqr
upper_bound = q3 + 1.5*iqr

# Outliers
outliers = data[(data < lower_bound) | (data > upper_bound)]

print(f"Minimum: {minimum}")
print(f"Q1 (25th percentile): {q1}")
print(f"Median: {median}")
print(f"Q3 (75th percentile): {q3}")
print(f"Maximum: {maximum}")
print(f"IQR: {iqr}")
print(f"Lower bound (no outlier): {lower_bound}")
print(f"Upper bound (no outlier): {upper_bound}")
print(f"Outliers:\n{outliers}")

**Correlation Analysis**

A correlation heatmap reveals relationships between variables. Because the features come from PCA, correlations tend to be low, but some components may still show meaningful relationships with the target. This helps identify potential predictors and understand the feature space.


In [ ]:
plt.figure(figsize=(16,12))
sns.heatmap(df.corr(), cmap='coolwarm', linewidths=0.5)
plt.title("Correlation Heatmap")
plt.show()


In [ ]:
# Matriz of correlations in numbers
correlation_matrix = df.corr()

# Show all matriz
print(correlation_matrix)

# 'Amount' column:
print(correlation_matrix['Amount'])


**Comparison of Transaction Amount Between Fraud and Non-Fraud**

This boxplot compares the spending patterns of fraudulent vs non-fraudulent transactions. Fraud cases typically show different spending behaviours, which helps identify potential predictive patterns.


In [ ]:
plt.figure(figsize=(6,4))
sns.boxplot(data=df, x='Class', y='Amount')
plt.title("Amount Distribution by Class")
plt.show()


In [ ]:
grouped = df.groupby('Class')['Amount']

for cls, amounts in grouped:
    minimum = np.min(amounts)
    q1 = np.percentile(amounts, 25)
    median = np.median(amounts)
    q3 = np.percentile(amounts, 75)
    maximum = np.max(amounts)
    iqr = q3 - q1
    lower_bound = q1 - 1.5*iqr
    upper_bound = q3 + 1.5*iqr
    outliers = amounts[(amounts < lower_bound) | (amounts > upper_bound)]

    print(f"\nClass {cls}:")
    print(f"Minimum: {minimum}")
    print(f"Q1 (25th percentile): {q1}")
    print(f"Median: {median}")
    print(f"Q3 (75th percentile): {q3}")
    print(f"Maximum: {maximum}")
    print(f"IQR: {iqr}")
    print(f"Lower bound (no outlier): {lower_bound}")
    print(f"Upper bound (no outlier): {upper_bound}")
    print(f"Number of outliers: {len(outliers)}")

**Transaction Time Analysis**

Credit card fraud often follows temporal patterns (e.g., occurring at night, during peak shopping times, or in bursts). The following plot examines whether frauds occur at specific times of the day.

**Interpretation:**

- Fraudulent transactions occur **earlier on average** (mean ≈ 80,747 seconds) compared to legitimate transactions (mean ≈ 94,838 seconds).
- The median time for fraud (≈ 75,569 seconds) is also lower, indicating a tendency for fraud to cluster in earlier intervals.
- Both classes exhibit high variability, but fraud shows slightly more spread, suggesting it occurs in bursts.
- While legitimate transactions appear more uniformly distributed across time, fraudulent transactions show **peaks in specific time windows**, indicating concentrated activity.

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(data=df, x='Time', hue='Class', bins=50, kde=True)
plt.title("Transaction Time Distribution by Class")
plt.show()


In [ ]:
# Filtramos por clase
class_0 = df[df['Class'] == 0]['Time']
class_1 = df[df['Class'] == 1]['Time']

# Definimos los bins (igual que en tu histplot)
bins = 50

# Calculamos histograma para cada clase
counts_0, bin_edges = np.histogram(class_0, bins=bins)
counts_1, _ = np.histogram(class_1, bins=bins)

# Mostramos en un DataFrame
import pandas as pd
hist_data = pd.DataFrame({
    'Bin_start': bin_edges[:-1],
    'Bin_end': bin_edges[1:],
    'Class_0_count': counts_0,
    'Class_1_count': counts_1
})

print(hist_data)

**Summary of Exploratory Data Analysis (EDA)**

The exploratory data analysis conducted on the credit card fraud dataset reveals several important insights that directly inform the subsequent preprocessing and modelling stages:

**Dataset completeness:**  
The dataset contains 284,807 transactions and no missing values, allowing for immediate progression to preprocessing without the need for imputation.

**Severe class imbalance:**  
Fraudulent transactions represent only 0.17% of all observations. This extreme imbalance confirms the need for specialised resampling techniques such as SMOTE, undersampling, or cost-sensitive learning to avoid biased model performance.

**Behaviour of PCA-derived features:**  
Features V1–V28, produced through PCA transformation, exhibit near-zero means and unit variance as expected. Their distributions are centred around zero but often highly skewed, reflecting how PCA compresses information from the original feature space.

**Transaction Amount characteristics:**  
The `Amount` variable shows heavy right skewness, with many low-value transactions and a small number of extremely large values.  

- Overall:
  - Minimum: 0.0
  - Q1 (25th percentile): 5.6
  - Median: 22.0
  - Q3 (75th percentile): 77.16
  - Maximum: 25,691.16
  - IQR: 71.57
  - Lower bound (no outlier): -101.75
  - Upper bound (no outlier): 184.51
  - Number of outliers: 31,904

- By class:
  - **Class 0 (Non-fraud)**
    - Maximum: 25,691.16
    - Outliers: 31,862
  - **Class 1 (Fraud)**
    - Maximum: 2,125.87
    - Outliers: 69  

These outliers may disproportionately influence distance-based and linear models unless scaling or transformation is applied.

**Temporal patterns in fraudulent activity:**  
Fraudulent transactions display distinct time-based patterns, occurring earlier on average and showing evidence of clustering in specific time windows. This suggests that engineered temporal features could improve classification performance.

**Correlation structure:**  
Due to the PCA transformation, correlations between features are generally weak. However, a few components show moderate association with the target variable, indicating potential predictive value despite the overall decorrelation imposed by PCA.

**Overall conclusion:**  
The EDA highlights the dataset’s complexity, the importance of handling class imbalance, and the potential usefulness of scaling, feature engineering, and resampling. Outlier handling, temporal features, and feature transformations will be key steps in Task 3.


# **Task 3 – Data Pre-processing**

This section details the preprocessing steps applied to the credit card fraud dataset.  
The objective is to clean and prepare the data for machine learning models while addressing class imbalance, scaling features, and applying feature engineering.  

**Overview**

The preprocessing workflow includes:

1. Handling missing values and duplicates  
2. Handling outliers in 'Amount'
3. Feature scaling  
4. Handling class imbalance (SMOTE)  
5. Feature engineering (temporal and amount features)  
6. Train/test splitting  

This ensures the dataset is ready for modelling, avoids data leakage, and maximises predictive performance.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)

**Handling Missing Values and Duplicates**

- Dataset contains **no missing values**, so no imputation is required.  
- Checked for duplicates to avoid bias in model training.  
- All features are numeric, compatible with most machine learning algorithms.


In [ ]:
# Make a safe copy to avoid SettingWithCopyWarning
df = df.copy()
# Check for duplicates
duplicates = df.duplicated().sum()
print("Number of duplicate rows:", duplicates)


- **Duplicates detected:** 1,081 rows. Removing duplicates ensures model training is not biased by repeated transactions.

In [ ]:
# Remove duplicates
df = df.drop_duplicates()
df.shape


**Handling Outliers in 'Amount'**

- Transaction `Amount` shows extreme right-skew with very large transactions.
- We use the IQR method to cap the outliers.

In [ ]:
# Calculate IQR and bounds for 'Amount'
Q1 = df['Amount'].quantile(0.25)
Q3 = df['Amount'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Cap outliers safely
df['Amount_capped'] = df['Amount'].clip(lower=lower_bound, upper=upper_bound)

# Quick check
df[['Amount', 'Amount_capped']].head()


In [ ]:
# Before capped
outliers_before = df[(df['Amount'] < lower_bound) | (df['Amount'] > upper_bound)].shape[0]

# After capped
outliers_after = df[(df['Amount_capped'] < lower_bound) | (df['Amount_capped'] > upper_bound)].shape[0]

print("Outliers before:", outliers_before)
print("Outliers after:", outliers_after)


In [ ]:
# Show Boxplot to check outliers removed
plt.figure(figsize=(8,4))
sns.boxplot(x=df['Amount_capped'])
plt.title("Boxplot of Transaction Amount")
plt.show()

In [ ]:
# Show Boxplot to check outliers removed
plt.figure(figsize=(6,4))
sns.boxplot(data=df, x='Class', y='Amount_capped')
plt.title("Amount Distribution by Class")
plt.show()

**Feature Engineering**

Even though PCA features are anonymised, additional derived features improve predictive power.

1. **Log-transform of 'Amount_capped'** to reduce skewness and limits outlier impact.

In [ ]:
# ===============================
# Feature Engineering (Before split)
# ===============================

# Log-transform of capped amount
df['Log_Amount'] = np.log1p(df['Amount_capped'])
df[['Amount_capped', 'Log_Amount']].head()


2. **Temporal features:** Fraud occurs in bursts at specific times  

- Derived 'Hour' from 'Time' in seconds  
- 'Day_period' splits day into 4 periods (night, morning, afternoon, evening)  
- Captures temporal patterns in fraud occurrence


In [ ]:
# Temporal features
df['Hour'] = (df['Time'] // 3600) % 24
df['Day_period'] = df['Hour'] // 6
df[['Time', 'Hour', 'Day_period']].head()


**Train/Test Split**

- Stratified split maintains class balance after SMOTE.  
- Random split ensures training and testing sets are representative.  
- Prevents model bias and allows reliable evaluation.


In [ ]:
from sklearn.model_selection import train_test_split
X = df.drop(['Class'], axis=1)
y = df['Class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Training set size:", X_train.shape)
print("Test set size:", X_test.shape)

**Summary of Pre-processing**

- Duplicates removed → 1,081 repeated transactions eliminated.
- Outliers capped → 'Amount' values beyond IQR bounds are limited to reduce extreme influence.
- Feature engineering → Log amount and temporal features capture patterns invisible in PCA features.
- Train/test split → Stratification maintains balanced distribution for reliable evaluation.

The dataset is now clean, balanced, scaled, and enriched, ready for modelling in Task 4.


# **Task 4 – Model Selection and Training**

This section presents the training of multiple machine learning models for credit card fraud detection.  
We compare **Logistic Regression, Naive Bayes, Random Forest, and XGBoost** with and without SMOTE, using F1-score as the primary metric due to class imbalance.


#**4.0 Handling Class Imbalance**

- Fraudulent transactions = **0.17%** → extreme class imbalance.  
- Applied **SMOTE**, generating synthetic fraud samples to balance the training set.  
- After SMOTE, both classes have **283,253 samples**.

**Balanced class distribution:**

{0: 226602, 1: 226602}  

This ensures models are not biased towards the majority class and improves fraud detection.



In [ ]:
from imblearn.over_sampling import SMOTE
# Apply SMOTE ONLY on training data
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

# Check new class distribution
unique, counts = np.unique(y_train_res, return_counts=True)
dict(zip(unique, counts))

**Feature Scaling**

In [ ]:
scale_cols = ['Amount_capped', 'Log_Amount']

scaler = StandardScaler()

X_train_res_scaled = X_train_res.copy()
X_test_scaled = X_test.copy()

# Fit scaler ONLY on training data
X_train_res_scaled[scale_cols] = scaler.fit_transform(
    X_train_res[scale_cols]
)

# Transform test data
X_test_scaled[scale_cols] = scaler.transform(
    X_test[scale_cols]
)

## **4.1 Logistic Regression**

Logistic Regression is a linear model suitable for binary classification. Class weighting is applied to address class imbalance.

**Interpretation:**
- Mean CV F1 = 0.981 → solid baseline.
- Class weighting balances the influence of rare fraud cases.


In [ ]:
# -------------------------------
# 4.1 Logistic Regression
# -------------------------------
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    random_state=42
)

lr.fit(X_train_res_scaled, y_train_res)

y_pred_lr = lr.predict(X_test_scaled)
f1_lr = f1_score(y_test, y_pred_lr)

print("Logistic Regression F1-score:", f1_lr)

**Interpretation:**  
- Logistic Regression provides a baseline linear model for fraud detection.  
- Cross-validation ensures stable performance estimation.  
- F1-score and classification report indicate initial model effectiveness for imbalanced classes.


## **2. Naive Bayes**

Naive Bayes is a probabilistic model assuming feature independence. Fast training makes it a good baseline.

**Interpretation:**

Mean CV F1 = 0.923 → lower than logistic regression, likely due to feature dependencies.

In [ ]:
# -------------------------------
# 4.2 Naive Bayes
# -------------------------------
from sklearn.naive_bayes import GaussianNB

nb = GaussianNB()
nb.fit(X_train_res_scaled, y_train_res)

y_pred_nb = nb.predict(X_test_scaled)
f1_nb = f1_score(y_test, y_pred_nb)

print("Naive Bayes F1-score:", f1_nb)

## **3. Random Forest**

Random Forest is a tree-based ensemble model that captures non-linear patterns. Hyperparameter tuning improves detection of minority classes.

**Interpretation:**  
- With SMOTE: F1 = 0.99988 → almost perfect CV performance.
- Without SMOTE: F1 = 0.8416 → slightly lower, but avoids synthetic data and still achieves strong results.

In [ ]:
# ===============================
# Random Forest WITHOUT SMOTE
# =============================== 17 minutes
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(
    n_estimators=200,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
f1_rf = f1_score(y_test, y_pred_rf)
print("Random Forest F1-score:", f1_rf)


## **4. XGBoost**

XGBoost is a gradient boosting model that handles non-linear relationships and imbalanced data efficiently.

**Interpretation:**
- With SMOTE: F1 = 0.99977 → near perfect.
- Without SMOTE: F1 = 0.7767 → slightly lower but avoids synthetic data and has high precision-recall balance.

In [ ]:
# ===============================
# XGBoost WITHOUT SMOTE
# =============================== 4 minutes
from xgboost import XGBClassifier
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
xgb = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    eval_metric='logloss'
)
xgb.fit(X_train, y_train)
y_pred_xgb = xgb.predict(X_test)
f1_xgb = f1_score(y_test, y_pred_xgb)
print("XGBoost F1-score:", f1_xgb)


**Summary Table**

In [ ]:
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Naive Bayes', 'Random Forest', 'XGBoost'],
    'F1-score': [f1_lr, f1_nb, f1_rf, f1_xgb]
})

results

## **Summary of Model Selection and Training**

**Summary:**
During Task 4, four different machine learning models were trained and evaluated for credit card fraud detection: Logistic Regression, Naive Bayes, Random Forest, and XGBoost.

- **Logistic Regression:** Provided a strong linear baseline, handling class imbalance effectively through class_weight. Training was fast and stable, with cross-validation F1-score around 0.981.
- **Naive Bayes:** Served as a probabilistic baseline, extremely fast to train. However, the independence assumption of features limited its performance on PCA-transformed variables.
- **Random Forest:** Captured non-linear interactions and complex patterns between features. Hyperparameter tuning with and without SMOTE showed that the model could adapt well to class imbalance. SMOTE slightly improved recall, while using class_weight alone achieved a strong balance between precision and recall.
- **XGBoost:** Training was slightly longer due to boosting iterations, but the model excelled at capturing non-linear relationships and handling imbalanced classes. Using scale_pos_weight instead of SMOTE maintained high precision and recall with fewer synthetic samples.

Training all four models gave insights into how different approaches handle extreme class imbalance, feature dependencies, and non-linear patterns. Ensemble models (RF and XGBoost) outperformed linear and probabilistic baselines in F1-score, while logistic regression offered interpretability and Naive Bayes offered speed.


#**Task 5 – Model Evaluation and Visualization**

This section evaluates the selected models on the test set and compares their performance using classification metrics and curves.

In [ ]:
# ===============================
# Task 5 – Model Evaluation
# ===============================

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    f1_score
)

In [ ]:
print("Logistic Regression\n")
print(classification_report(y_test, y_pred_lr))

print("Naive Bayes\n")
print(classification_report(y_test, y_pred_nb))

print("Random Forest\n")
print(classification_report(y_test, y_pred_rf))

print("XGBoost\n")
print(classification_report(y_test, y_pred_xgb))

**5.2 Confusion Matrices**


In [ ]:
models = {
    "Logistic Regression": y_pred_lr,
    "Naive Bayes": y_pred_nb,
    "Random Forest": y_pred_rf,
    "XGBoost": y_pred_xgb
}

plt.figure(figsize=(12,10))

for i, (name, y_pred) in enumerate(models.items(), 1):
    plt.subplot(2, 2, i)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(name)
    plt.xlabel("Predicted")
    plt.ylabel("Actual")

plt.tight_layout()
plt.show()


**5.3 ROC–AUC Analysis**


In [ ]:
# Probability predictions
y_prob_lr = lr.predict_proba(X_test_scaled)[:, 1]
y_prob_nb = nb.predict_proba(X_test_scaled)[:, 1]
y_prob_rf = rf.predict_proba(X_test)[:, 1]
y_prob_xgb = xgb.predict_proba(X_test)[:, 1]

# ROC-AUC scores
roc_scores = {
    "Logistic Regression": roc_auc_score(y_test, y_prob_lr),
    "Naive Bayes": roc_auc_score(y_test, y_prob_nb),
    "Random Forest": roc_auc_score(y_test, y_prob_rf),
    "XGBoost": roc_auc_score(y_test, y_prob_xgb)
}

roc_scores

**5.4 ROC Curves**

In [ ]:
plt.figure(figsize=(8,6))

for name, y_prob in zip(
    roc_scores.keys(),
    [y_prob_lr, y_prob_nb, y_prob_rf, y_prob_xgb]
):
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    plt.plot(fpr, tpr, label=f"{name}")

plt.plot([0,1], [0,1], linestyle='--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves Comparison")
plt.legend()
plt.show()

**Precision-Recall Curves**

In [ ]:
plt.figure(figsize=(8,6))

for name, y_prob in zip(
    roc_scores.keys(),
    [y_prob_lr, y_prob_nb, y_prob_rf, y_prob_xgb]
):
    precision, recall, _ = precision_recall_curve(y_test, y_prob)
    plt.plot(recall, precision, label=name)

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision–Recall Curves")
plt.legend()
plt.show()


**Model Comparison Summary**

In [ ]:
evaluation_summary = pd.DataFrame({
    "Model": ["Logistic Regression", "Naive Bayes", "Random Forest", "XGBoost"],
    "F1-score": [f1_lr, f1_nb, f1_rf, f1_xgb],
    "ROC-AUC": [
        roc_scores["Logistic Regression"],
        roc_scores["Naive Bayes"],
        roc_scores["Random Forest"],
        roc_scores["XGBoost"]
    ]
})

evaluation_summary

## **Summary of Model Evaluation and Visualizationg**

**Summary:**
Task 5 evaluated the four models on the test set using classification metrics, ROC curves, and Precision-Recall curves to compare real-world predictive performance.

- **Logistic Regression:** High precision for non-fraud transactions, but recall for fraud was low, highlighting limitations for extremely imbalanced datasets.
- **Naive Bayes:** Poor detection of fraud due to the feature independence assumption. Fast inference but low F1-score for fraud.
- **Random Forest:** Strong performance in both SMOTE and non-SMOTE versions. Without SMOTE, the model achieved a good precision-recall balance, reducing false positives while maintaining high detection of fraud.
- **XGBoost:** Performed slightly better without SMOTE, achieving the best trade-off between precision and recall, making it highly suitable for production. With SMOTE, recall improved slightly, but at the cost of more false positives.

Evaluation revealed that ensemble tree-based models (Random Forest and XGBoost) are more robust for fraud detection than linear or probabilistic models. SMOTE can improve recall for rare fraud cases, but careful consideration is needed to avoid excessive false positives. Visualizations like ROC and Precision-Recall curves confirmed the superior performance of RF and XGBoost, and highlighted the trade-offs between models.

In [ ]:
s